# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/i-am-gk/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Contract

**Unit of analysis:** One row represents one pseudonymized content item (page).

**Time window:** The starter dataset contains aggregated metrics over a trailing 90-day window ending at export time. The dataset is a snapshot rather than a daily time-series table.

This contract treats the page as the unit being analyzed and keeps the 90-day activity window attached to each page.

In [1]:
import pandas as pd
from pathlib import Path

# Load the starter dataset uploaded to the Colab session.
csv_path = Path("/content/content_refresh_anonymized.csv")

if not csv_path.exists():
    matches = list(Path("/content").rglob("content_refresh_anonymized.csv"))
    if matches:
        csv_path = matches[0]
    else:
        raise FileNotFoundError(
            "Upload content_refresh_anonymized.csv to the Colab Files panel first."
        )

df = pd.read_csv(csv_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unit of analysis: one row per content item")
print("Expected starter shape: approximately 30,000 rows × 44 columns")
print("\nFirst columns:", df.columns.tolist()[:10])

Rows: 30000
Columns: 44
Unit of analysis: one row per content item
Expected starter shape: approximately 30,000 rows × 44 columns

First columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count']


### Field contract

**Features:** `search_volume`, `competition`, `cpc`, `word_count`, `content_age_days`, `days_since_last_update`, `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`, `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, and `sessions_prev_30d`.

**Label / proxy:** `trend_direction` is the observed trend category used by the starter pipeline to derive `is_declining_label`. `trend_pct` is also label-derived and is therefore not a feature.

**Context:** `content_id`, `client_id`, `content_type`, `main_intent`, and `competition_level`. These fields help identify, group, describe, or interpret the content but are not used as learned numeric signals in this contract.

**Excluded:** `trend_pct` and `trend_direction` are excluded from features because they define the decline outcome. `provider_used` and `model_used` are excluded because they describe content-generation provenance rather than the search-performance signal being analyzed. `content_id` and `client_id` are identifiers and are used only for grouping, joins, or inspection.

In [2]:
feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]

label_cols = [
    "trend_direction",
    "trend_pct",
]

context_cols = [
    "content_id",
    "client_id",
    "content_type",
    "main_intent",
    "competition_level",
]

excluded_cols = [
    "provider_used",
    "model_used",
]

print("Feature fields:", len(feature_cols))
print("Label / proxy fields:", label_cols)
print("Context fields:", context_cols)
print("Excluded fields:", excluded_cols)

missing_from_dataset = [
    c for c in feature_cols + label_cols + context_cols + excluded_cols
    if c not in df.columns
]

print("\nMissing contract fields:", missing_from_dataset)

Feature fields: 22
Label / proxy fields: ['trend_direction', 'trend_pct']
Context fields: ['content_id', 'client_id', 'content_type', 'main_intent', 'competition_level']
Excluded fields: ['provider_used', 'model_used']

Missing contract fields: []


### Verification

The checks below verify the dataset grain, row/column counts, missingness, and the values used by the contract.

The starter dataset should contain approximately 30,000 page-level rows. Missingness is expected in some keyword and content-property fields, so missing values are measured rather than silently treated as zero.

In [3]:
# 1. Basic shape
print("=== DATASET SHAPE ===")
print("Rows:", len(df))
print("Columns:", len(df.columns))

# 2. Grain check
print("\n=== GRAIN CHECK ===")
duplicate_ids = df["content_id"].duplicated().sum()
print("Duplicate content_id values:", duplicate_ids)

# 3. Client count
print("\n=== CLIENT COUNT ===")
print("Unique clients:", df["client_id"].nunique())

# 4. Missingness for important fields
check_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "trend_direction",
    "trend_pct",
]

missing_table = pd.DataFrame({
    "missing_n": df[check_cols].isna().sum(),
    "missing_pct": (df[check_cols].isna().mean() * 100).round(2),
})

print("\n=== MISSINGNESS ===")
display(missing_table)

# 5. Window-related summary
print("\n=== 90-DAY WINDOW CHECK ===")
window_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
]

display(df[window_cols].describe().T)

# 6. Label distribution
print("\n=== TREND DISTRIBUTION ===")
display(df["trend_direction"].value_counts(dropna=False))

=== DATASET SHAPE ===
Rows: 30000
Columns: 44

=== GRAIN CHECK ===
Duplicate content_id values: 0

=== CLIENT COUNT ===
Unique clients: 32

=== MISSINGNESS ===


,missing_n,missing_pct
search_volume,2468,8.23
competition,2468,8.23
cpc,2468,8.23
word_count,7699,25.66
days_since_last_update,0,0.00
impressions_90d,0,0.00
clicks_90d,0,0.00
sessions_90d,0,0.00
trend_direction,0,0.00
trend_pct,3388,11.29



=== 90-DAY WINDOW CHECK ===


,count,mean,std,min,25%,50%,75%,max
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.0,3615.25,517715.0
clicks_90d,30000.0,16.097333,75.076958,0.0,0.0,1.0,7.00,4178.0
sessions_90d,30000.0,37.066633,107.069131,1.0,2.0,7.0,27.00,4345.0
days_with_impressions,30000.0,61.454033,32.689245,1.0,31.0,81.0,88.00,88.0
days_with_sessions,30000.0,13.096333,17.307759,1.0,2.0,6.0,16.00,90.0



=== TREND DISTRIBUTION ===


,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


### Data limits

This starter dataset is a snapshot with aggregated trailing-90-day metrics, so it cannot by itself establish causal effects or show the exact day-to-day sequence of events.

The data also has systematic missingness. Keyword-context fields can be unavailable for some content types, and content-property fields such as `word_count` can be missing. Missingness should therefore be measured rather than blindly interpreted as zero.

The trend fields describe the observed comparison used to define the decline label. Because they are label-derived, they must not be used as model features.

The dataset also does not provide client names, URLs, private queries, or other identifying information, so conclusions are limited to the anonymized fields available here.

Results from this dataset should be described as observed, measured, or directional decision-support rather than as causal claims.

In [4]:
print("=== DATA LIMIT CHECKS ===")

print("Snapshot rows:", len(df))
print("Unique clients:", df["client_id"].nunique())

print("\nContent types:")
display(df["content_type"].value_counts(dropna=False))

print("\nKeyword-data availability:")
keyword_cols = ["search_volume", "competition", "cpc"]
keyword_available = df[keyword_cols].notna().any(axis=1)
print("Rows with at least some keyword data:", int(keyword_available.sum()))
print("Rows without keyword data:", int((~keyword_available).sum()))

print("\nWord-count availability:")
print("Rows with word_count:", int(df["word_count"].notna().sum()))
print("Rows missing word_count:", int(df["word_count"].isna().sum()))

print("\nLabel-derived fields are excluded from features:")
print("trend_direction =", "excluded")
print("trend_pct =", "excluded")

=== DATA LIMIT CHECKS ===
Snapshot rows: 30000
Unique clients: 32

Content types:


,count
content_type,
keyword article,27207
feedly article,2096
comparison article,697



Keyword-data availability:
Rows with at least some keyword data: 27532
Rows without keyword data: 2468

Word-count availability:
Rows with word_count: 22301
Rows missing word_count: 7699

Label-derived fields are excluded from features:
trend_direction = excluded
trend_pct = excluded


## Self-check

- [x] Every section is filled with Markdown reasoning and supporting code.
- [x] The notebook is designed to run top to bottom without requiring private data.
- [x] No client names, URLs, or private search queries are used.
- [x] The contract distinguishes features, label/proxy, context, and excluded fields.
- [x] `trend_direction` and `trend_pct` are excluded from features because they define the decline outcome.
- [x] IDs are treated as context rather than model features.
- [x] Missingness is measured instead of blindly converting missing values to zero.
- [x] Claims use careful language such as observed, measured, and directional decision-support.